# 1. Free propagation, flux conservation, and continuum convergence

Read [Chapters 2–3](../03-scattering.md). We use the actual finite-difference
solver and the library's independently coded continuum references. The mass
ratio is illustrative. Phases are referred to the two endpoints, so a free
device has $t(E)=\exp(ik(E)L)$, not one. Here $k$ is inverse nm, $L$ is nm,
and the phase and amplitudes are dimensionless.

The verification criteria below are regression guards for this specified grid
study, based on the existing scattering convergence tests. They are not physical
laws or universal accuracy guarantees. All arrays, even unfavorable errors,
are retained. Run from the repository or a notebook subdirectory.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import numpy as np
import os, tempfile
if "MPLCONFIGDIR" not in os.environ:
    _mpl_cache=tempfile.TemporaryDirectory(prefix="qwave-teaching-mpl-")
    os.environ["MPLCONFIGDIR"]=_mpl_cache.name
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/qwave").is_dir())
sys.path.insert(0, str(ROOT / "docs/derivations"))
from verification import VerificationBundle
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": .2})

CONFIG = {'seeds': [], 'effective_mass_ratio': 0.067, 'leads_mev': [0.0, 0.0], 'domain_nm': [-40.0, 40.0], 'grid_points': [321, 641, 1281], 'check_energies_mev': [5.0, 10.0, 20.0, 39.0, 40.0, 41.0, 60.0, 100.0], 'plot_energy_grid_mev': [5.0, 100.0, 241], 'barrier': {'kind': 'rectangular', 'offset_mev': 0.0, 'height_mev': 40.0, 'center_nm': 0.0, 'width_nm': 10.0}, 'approximations': ['centered finite difference', 'exact discrete semi-infinite leads', 'cell-average rectangular edges', 'float64/complex128'], 'criteria': {'conservation_atol': 1e-10, 'free_phase_atol': 1e-10, 'reference_probability_atol': 1e-12, 'refinement_ratio_bounds': [3.5, 4.5], 'fine_t_atol': 0.0011, 'fine_T_atol': 4e-05}}

output = Path(globals().get("RUN_OUTPUT", ROOT / "results" /
              ("learning-01-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))))
bundle = VerificationBundle(output, CONFIG, scope='educational physical-model benchmarks, not an optimized device')
print(json.dumps(CONFIG, indent=2))


In [ ]:
from qwave.scattering import solve_scattering, transmission_phase
from qwave.potentials import sample_potential
from qwave.references import rectangular_probability, rectangular_transmission_amplitude
from scipy.constants import hbar, m_e, elementary_charge

energy = np.asarray(CONFIG["check_energies_mev"])
mass = CONFIG["effective_mass_ratio"]
length = CONFIG["domain_nm"][1]-CONFIG["domain_nm"][0]
reference_args = dict(height_mev=CONFIG["barrier"]["height_mev"], width_nm=CONFIG["barrier"]["width_nm"], effective_mass_ratio=mass)
exact_T = rectangular_probability(energy, **reference_args)
exact_t = rectangular_transmission_amplitude(energy, **reference_args, reference_length_nm=length)
continuum_k = np.sqrt(2*mass*m_e*energy*elementary_charge*1e-3)/hbar*1e-9
rows = []
for count in CONFIG["grid_points"]:
    x = np.linspace(*CONFIG["domain_nm"], count)
    potential = sample_potential(x, CONFIG["barrier"])
    free = solve_scattering(x, 0*x, energy, effective_mass_ratio=mass)
    result = solve_scattering(x, potential, energy, effective_mass_ratio=mass)
    rows.append(dict(dx_nm=result.spacing_nm,
        max_free_discrete_t_error=float(max(abs(free.t-np.exp(1j*free.k_left_per_nm*length)))),
        max_free_continuum_t_error=float(max(abs(free.t-np.exp(1j*continuum_k*length)))),
        max_barrier_t_error=float(max(abs(result.t-exact_t))),
        max_barrier_T_error=float(max(abs(result.T-exact_T))),
        max_conservation_error=float(max(result.conservation_error)),
        max_relative_residual=float(max(result.relative_residual))))
    bundle.arrays(f"grid-{count}", units={"x_nm":"nm","V_mev":"meV","energy_mev":"meV",
        "t":"1 complex","r":"1 complex","T":"1","R":"1","free_t":"1 complex",
        "reference_t":"1 complex","reference_T":"1"}, x_nm=x,V_mev=potential,energy_mev=energy,
        t=result.t,r=result.r,T=result.T,R=result.R,free_t=free.t,reference_t=exact_t,reference_T=exact_T)
bundle.table("convergence", rows)
print(json.dumps(rows, indent=2))


The free discrete phase is checked within the same discrete dispersion.
The separate continuum comparison uses $k=\sqrt{2m^*E}/\hbar$, with energies
converted to joules and lengths to nm. These are distinct checks: conserving
flux on a coarse grid does not imply the continuum phase is accurate.
The barrier energies include below, at, and above its top.

In [ ]:
fine = rows[-1]
criteria = CONFIG["criteria"]
ref_error = float(max(abs(abs(exact_t)**2-exact_T)))
bundle.check("independent continuum formulas agree", ref_error < criteria["reference_probability_atol"],
    measured=ref_error,criterion="Absolute T difference < 1e-12; independent real and complex formulas")
bundle.check("free discrete complex phase", max(row["max_free_discrete_t_error"] for row in rows)<criteria["free_phase_atol"],
    measured=[row["max_free_discrete_t_error"] for row in rows],criterion="Fixed-grid roundoff guard 1e-10")
bundle.check("lossless probability conservation", max(row["max_conservation_error"] for row in rows)<criteria["conservation_atol"],
    measured=[row["max_conservation_error"] for row in rows],criterion="T+R differs from one by < 1e-10")
errors = np.array([row["max_barrier_t_error"] for row in rows])
ratios = errors[:-1]/errors[1:]
bundle.check("complex barrier amplitude refinement", np.all((ratios>3.5)&(ratios<4.5)),
    measured=ratios,criterion="Halving dx gives error ratio in [3.5,4.5], a resolution-specific second-order check")
bundle.check("fine continuum comparison", fine["max_barrier_t_error"]<criteria["fine_t_atol"] and fine["max_barrier_T_error"]<criteria["fine_T_atol"],
    measured={"t_error":fine["max_barrier_t_error"],"T_error":fine["max_barrier_T_error"]},
    criterion="At dx=0.0625 nm, max |delta t|<0.0011 and max |delta T|<0.00004 (existing validation guards)")

e_low,e_high,e_points = CONFIG["plot_energy_grid_mev"]
ep = np.linspace(e_low,e_high,int(e_points))
plot_result = solve_scattering(x,potential,ep,effective_mass_ratio=mass)
tp = rectangular_transmission_amplitude(ep,**reference_args,reference_length_nm=length)
bundle.arrays("spectrum",units={"energy_mev":"meV","t":"1 complex","r":"1 complex","T":"1","R":"1","phase_rad":"rad","continuum_t":"1 complex"},
    energy_mev=ep,t=plot_result.t,r=plot_result.r,T=plot_result.T,R=plot_result.R,
    phase_rad=transmission_phase(plot_result.t),continuum_t=tp)
fig,axs=plt.subplots(2,2,figsize=(10,7),layout="constrained")
axs[0,0].plot(x,potential);axs[0,0].set(xlabel="Position (nm)",ylabel="Potential energy (meV)")
axs[0,1].plot(ep,plot_result.T,label="Discrete T");axs[0,1].plot(ep,abs(tp)**2,"--",label="Continuum T")
axs[0,1].set(xlabel="Energy (meV)",ylabel="Transmission probability (1)");axs[0,1].legend()
axs[1,0].plot(ep,plot_result.R);axs[1,0].set(xlabel="Energy (meV)",ylabel="Reflection probability (1)")
axs[1,1].plot(ep,transmission_phase(plot_result.t),label="Discrete")
axs[1,1].plot(ep,transmission_phase(tp),"--",label="Continuum")
axs[1,1].set(xlabel="Energy (meV)",ylabel="Unwrapped transmission phase (rad)");axs[1,1].legend()
bundle.figure(fig,"scattering","Rectangular barrier; complex phase between -40 and +40 nm planes, including free padding. Finite-grid versus continuum comparison; no optimization claim.")
fig,ax=plt.subplots(figsize=(6,4),layout="constrained")
spacing=np.array([row["dx_nm"] for row in rows])
ax.loglog(spacing,errors,"o-",label="Measured complex t error")
ax.loglog(spacing,errors[0]*(spacing/spacing[0])**2,"--",label="Second-order guide anchored to first error")
ax.set(xlabel="Grid spacing (nm)",ylabel="Maximum complex amplitude error (1)");ax.legend()
bundle.figure(fig,"scattering-convergence","Three actual grids at fixed physical dimensions and energies. The dashed slope guide is labeled and is not another measured data series.")
bundle.finish()
